In [1]:
import os
import shutil
import pathlib
from datetime import datetime
from PIL import Image
from PIL.ExifTags import TAGS
from pathlib import Path

In [49]:
# source_base = "C:/Users/laram/OneDrive/Pictures/"
# source_base = "C:/Users/laram/OneDrive/Documents/Gifts & Crafts"
source_base = "C:/Users/laram/Pictures/Dropbox Copy - Copy/"
destination_base = "C:/Users/laram/Pictures/renamed/"
destination_base = "C:/Users/laram/Pictures/renamed - testing/"

In [50]:
def copy_file(source, suffix=''):
    # this function successfully copies a photo from the source path to the destination path

    global successful_copies
    
    if get_datetime_string(source, suffix):

        file_path = destination_base + get_datetime_string(source, suffix)

        try:
            shutil.copyfile(source, file_path)
            successful_copies += 1
            successful_copies_list.append(source)
#             os.remove(source)

        # If source and destination are same
        except shutil.SameFileError:
            print("Source and destination represents the same file: " + file_path)

        # If destination is a directory.
        except IsADirectoryError:
            print("Destination is a directory: " + file_path)

        # If there is any permission issue
        except PermissionError:
            print("Permission denied to path: " + file_path)

        except Exception as e: print(e)

        # For other errors
        except:
            print("Error occurred while copying file path: " + file_path)


def get_exif_data(image_path):
    image = Image.open(image_path)
    exif_data = image._getexif()

    if exif_data is not None:
        exif_info = {}
        for tag, value in exif_data.items():
            tag_name = TAGS.get(tag, tag)
            exif_info[tag_name] = value
        return exif_info
    else:
        return None


def get_datetime_string(image_path, suffix):

    global failed_exif_files
    global none_exif_count_files
    global failed_datetime_files


    failed_to_find_name = False


    try:
        exif_data = get_exif_data(image_path)
    except:
        exif_data = None
        failed_exif_files.append(image_path)
        failed_to_find_name = True
        #return image_path.split('/')[-1]

    if exif_data is None:
        none_exif_count_files.append(image_path)
        failed_to_find_name = True
        #return image_path.split('/')[-1]

    try:
        datetime_value = exif_data['DateTime']
    except:
        failed_datetime_files.append(image_path)
        failed_to_find_name = True
        #return image_path.split('/')[-1]

    if not failed_to_find_name:
        date = datetime_value.split()[0]
        date = "-".join(date.split(":"))
        time = "".join(datetime_value.split()[1].split(":"))
        datetime = date + " " + time 

    elif is_datetime_value_in_filename(image_path.split('/')[-1]):
        datetime = return_datetime_value_from_file_name(filename)

    else:
        other_failed_files.append(image_path)
        return None
#         return Path(image_path).name
#         return image_path.split('/')[-1]


    counter = 0
    datetime_counter = datetime + str(counter)

    while datetime_counter + "." + image_path[-4:] in os.listdir(destination_base):
        counter += 1
        datetime_counter = datetime + str(counter)

    datetime_filename = datetime_counter + suffix + "." + image_path.split(".")[-1]

    return datetime_filename

def is_datetime_value_in_filename(filename):
    try:
        original_file_date = int(original_file_name[4:12])
        original_file_time = int(original_file_name[13:19])

    except:
        return False

    date_year = int(original_file_name[4:8])
    date_month = int(original_file_name[8:10])
    date_day = int(original_file_name[10:12])
    time_hour = int(original_file_name[13:19])
    time_minute = int(original_file_name[13:19])
    time_second = int(original_file_name[13:19])

    if date_year > 1900:
        if date_month > 0 and date_month <= 12:
            if date_day > 0 and date_day <= 31:
                if time_hour >= 0 and time_hour <= 23:
                    if time_minute >= 0 and time_minute <= 59:
                        if time_second >= 0 and time_second <= 59:
                            return True
    else:
        return False

def return_datetime_value_from_file_name(filename):

    date_year = filename[4:8] + '-'
    date_month = filename[8:10] + '-'
    date_day = filename[10:12] + ' '
    time_hour = filename[13:19]
    time_minute = filename[13:19]
    time_second = filename[13:19]

    file_name = date_year + date_month + date_day + time_hour + time_minute + time_second


In [51]:
# Adjusted code from above to go through files recursively and add a description list

successful_copies = 0
successful_copies_list = []
failed_exif_files = []
none_exif_count_files = []
failed_datetime_files = []
other_failed_files = []

folder = Path(source_base)
count_files = 0
count_files_list = []
count_jpgs = 0
count_jpgs_list = []
unmoved_nonjpgs = []

for file in folder.rglob("*"):
    
    file_source_path = str(file)
    file_extension = file.suffix
    file_name = file.stem
    
    ## Ignore if already timestamped and just move as named:
    try:
        datetime.strptime(file_name[:17], "%Y-%m-%d_%H%M%S")
        destination_file_path = destination_base + 'already/' + file.name
        shutil.copyfile(file_source_path, destination_file_path)
        continue
        
    except ValueError:
        pass
    
    if file.is_file():
        count_files += 1
        count_files_list.append(file_source_path)
        
        if file_extension.upper() in ['.JPG', '.JPEG']:
            count_jpgs += 1
            count_jpgs_list.append(file_source_path)
            added_suffix = ' - ' + file.parent.name
            copy_file(file_source_path, added_suffix)
        else:
            unmoved_nonjpgs.append(file_source_path)


        
print('Found ' + str(count_files) + ' files in this directory.')
print('Found ' + str(count_jpgs) + ' JPG files in this directory.')
print('Successfully Copied ' + str(successful_copies) +
      ' files to the destination directory.')
print(str(len(failed_exif_files)) + ' files did not have any EXIF data.')
print(str(len(none_exif_count_files)) + ' files EXIF data was empty.')
print(str(len(failed_datetime_files)) +
      ' files had EXIF data but no DateTime data.')
print(str(len(other_failed_files)) + ' files failed due to other EXIF data errors')

Found 3459 files in this directory.
Found 1647 JPG files in this directory.
Successfully Copied 759 files to the destination directory.
0 files did not have any EXIF data.
68 files EXIF data was empty.
888 files had EXIF data but no DateTime data.
888 files failed due to other EXIF data errors


In [ ]:
# I counted 1647 .jpg files

In [ ]:
# Found 3459 files == correct
# Found 1647 JPG files == correct
# Copied 759 files == *WRONG* -> only copied 717 -> missing 42 photos
# The "68 files" with empty exif data is included in the 888 files that had exif data and no datetime

In [54]:
# for each file in all files, which file is not in successful/failed folders
successes = []
failed = []

unaccounted_for = []

for file in count_files_list:
    
    if file in successful_copies_list:
        successes.append(file)
        
    elif file in failed_exif_files OR file in none_exif_count_files OR file in failed_datetime_files OR file in other_failed_files:
        failed.append(file)
    
    else:
        unaccounted_for.append(file)

SyntaxError: invalid syntax (1251725860.py, line 12)